In [ ]:
import os
import pickle
import mlflow
import boto3
import numpy as np
import pandas as pd
from io import StringIO
from lightfm import LightFM
from lightfm.data import Dataset
from lightfm.evaluation import precision_at_k

# === S3 Utility ===
s3 = boto3.client("s3", region_name="us-east-1")

def read_csv_from_s3(bucket: str, key: str) -> pd.DataFrame:
    response = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(StringIO(response["Body"].read().decode("utf-8")))

# === Load Data ===
stock_df = read_csv_from_s3("etoro-data-prj", "stocks/item_features.csv")
user_df = read_csv_from_s3("etoro-data-prj", "investor/csv/etoro_investors.csv")
inter_df = pd.read_csv("filtered_user_holdings.csv")

# === Clean & Preprocess ===
inter_df["invested"] = (
    inter_df["invested"]
    .astype(str).str.replace(",", "")
    .str.replace("%", "").str.replace("-", "").str.strip()
)
inter_df["invested"] = pd.to_numeric(inter_df["invested"], errors="coerce").fillna(0)

popular = inter_df["symbol"].value_counts()
inter_df = inter_df[inter_df["symbol"].isin(popular[popular >= 50].index)]

stock_df.rename(columns={
    "Symbol": "symbol",
    "sector": "sector",
    "industry": "industry",
    "asset_type": "asset_type"
}, inplace=True)

merged_df = pd.merge(inter_df, stock_df, on="symbol", how="left")
triples = list(zip(merged_df["username"], merged_df["symbol"], merged_df["invested"] / 100.0))

# === Feature Engineering Functions ===
def categorize_marketcap(val):
    if pd.isna(val): return "marketcap:unknown"
    elif val < 1e9: return "marketcap:small"
    elif val < 1e10: return "marketcap:mid"
    else: return "marketcap:large"

def categorize_div_yield(val):
    if pd.isna(val): return "dividend:none"
    elif val < 1: return "dividend:low"
    elif val < 3: return "dividend:mid"
    else: return "dividend:high"

def categorize_pb(val):
    if pd.isna(val): return "pb:unknown"
    elif val < 1: return "pb:undervalued"
    elif val > 5: return "pb:overvalued"
    else: return "pb:fair"

def categorize_volume(val):
    if pd.isna(val): return "volume:unknown"
    elif val < 1e6: return "volume:low"
    elif val < 1e7: return "volume:mid"
    else: return "volume:high"

def price_volatility(row):
    if row['fiftyTwoWeekHigh'] == 0: return "volatility:unknown"
    ratio = (row['fiftyTwoWeekHigh'] - row['fiftyTwoWeekLow']) / row['fiftyTwoWeekHigh']
    if ratio < 0.2: return "volatility:low"
    elif ratio < 0.4: return "volatility:medium"
    else: return "volatility:high"

# === Build Item Features ===
item_features_dict = {}
unique_items = merged_df.drop_duplicates("symbol")
for _, row in unique_items.iterrows():
    item_features_dict[row["symbol"]] = [
        f"asset_type:{row['asset_type']}",
        f"sector:{row['sector']}" if pd.notna(row['sector']) else "sector:unknown",
        f"industry:{row['industry']}" if pd.notna(row['industry']) else "industry:unknown",
        categorize_marketcap(row['marketCap']),
        categorize_div_yield(row['dividendYield']),
        categorize_pb(row['priceToBook']),
        categorize_volume(row['averageVolume']),
        price_volatility(row)
    ]

# === Build User Features ===
def normalize_risk(r):
    try:
        r = int(r)
        return "low" if r <= 3 else "moderate" if r <= 6 else "high"
    except:
        return "moderate"

user_features_dict = {
    row["username"]: [f"risk:{normalize_risk(row['risk'])}"]
    for _, row in user_df.iterrows()
}

def build_user_interest_features(interactions_df, stock_df, user_features_dict, threshold=0.3):
    merged = pd.merge(interactions_df, stock_df, on="symbol", how="left")
    for username, group in merged.groupby("username"):
        features = []
        sector_counts = group["sector"].value_counts(normalize=True)
        for sector, pct in sector_counts.items():
            if pd.notna(sector) and pct >= threshold:
                features.append(f"interest_sector:{sector.lower()}")  # 🔽
        type_counts = group["asset_type"].value_counts(normalize=True)
        for asset_type, pct in type_counts.items():
            if pd.notna(asset_type) and pct >= threshold:
                features.append(f"interest_asset_type:{asset_type.lower()}")  # 🔽
        if username in user_features_dict:
            user_features_dict[username].extend(f for f in features if f not in user_features_dict[username])
        else:
            user_features_dict[username] = features

build_user_interest_features(inter_df, stock_df, user_features_dict)

# === LightFM Dataset ===
user_features_dict["__cold_user__"] = user_features_dict.get("__cold_user__", [
    "risk:moderate",
    "interest_asset_type:etf",
    "interest_sector:technology"
])

# === Dataset 設定 ===
dataset = Dataset()

# 加入 __cold_user__ 到使用者列表
all_users = set(merged_df["username"]).union(user_features_dict.keys())

dataset.fit(
    users=all_users,
    items=merged_df["symbol"],
    user_features=[f for fs in user_features_dict.values() for f in fs],
    item_features=[f for fs in item_features_dict.values() for f in fs]
)

(interactions, _) = dataset.build_interactions(triples)
user_features = dataset.build_user_features(user_features_dict.items())
item_features = dataset.build_item_features(item_features_dict.items())

# === MLflow Setup ===
mlflow.set_tracking_uri("http://localhost:5001")
mlflow.set_experiment("lightfm-recommender")

# === Model Training ===
k_list = [10, 20, 50]
lr_list = [0.005, 0.01, 0.05]

best_model = None
best_score = 0
best_config = {}

for k in k_list:
    for lr in lr_list:
        with mlflow.start_run(run_name=f"k={k}_lr={lr}"):
            print(f"Training model: k={k}, lr={lr}")
            model = LightFM(no_components=k, learning_rate=lr, loss="warp")
            model.fit(interactions, user_features=user_features, item_features=item_features, epochs=20, num_threads=4)

            score = precision_at_k(model, interactions, user_features=user_features, item_features=item_features, k=5).mean()

            mlflow.log_param("no_components", k)
            mlflow.log_param("learning_rate", lr)
            mlflow.log_metric("precision_at_5", score)

            if score > best_score:
                best_score = score
                best_model = model
                best_config = {"no_components": k, "learning_rate": lr}

# === Save Best Model ===
with open("best_lightfm_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

# save dataset 
with open("lightfm_dataset.pkl", "wb") as f:
    pickle.dump(dataset, f)

with open("item_features_dict.pkl", "wb") as f:
    pickle.dump(item_features_dict, f)
    
with open("stock_df.pkl", "wb") as f:
    pickle.dump(stock_df, f)

# === Log Best Summary ===
with mlflow.start_run(run_name="summary"):
    mlflow.log_params(best_config)
    mlflow.log_metric("best_precision_at_5", best_score)
    mlflow.log_artifact("best_lightfm_model.pkl")
    mlflow.log_artifact("lightfm_dataset.pkl")
    mlflow.log_artifact("item_features_dict.pkl")
    mlflow.log_artifact("lightfm_dataset.pkl")


Training model: k=10, lr=0.005
🏃 View run k=10_lr=0.005 at: http://localhost:5001/#/experiments/15/runs/6851aa5994d54909854ef911fd4a851e
🧪 View experiment at: http://localhost:5001/#/experiments/15
Training model: k=10, lr=0.01
🏃 View run k=10_lr=0.01 at: http://localhost:5001/#/experiments/15/runs/926f6d103f3d4957849b84e54528c93e
🧪 View experiment at: http://localhost:5001/#/experiments/15
Training model: k=10, lr=0.05
🏃 View run k=10_lr=0.05 at: http://localhost:5001/#/experiments/15/runs/92f386aba1074c2493918b91c810f8e8
🧪 View experiment at: http://localhost:5001/#/experiments/15
Training model: k=20, lr=0.005
🏃 View run k=20_lr=0.005 at: http://localhost:5001/#/experiments/15/runs/aa49371096e64348aae69cf33aaba056
🧪 View experiment at: http://localhost:5001/#/experiments/15
Training model: k=20, lr=0.01
🏃 View run k=20_lr=0.01 at: http://localhost:5001/#/experiments/15/runs/b5e8bf35dcbc44f5aa5ba73a142ef491
🧪 View experiment at: http://localhost:5001/#/experiments/15
Training model: 

In [232]:
# 擷取所有 interest_sector 的值
all_sectors = set()

for features in user_features_dict.values():
    for feat in features:
        if feat.startswith("interest_sector:"):
            sector = feat.split("interest_sector:")[1]
            all_sectors.add(sector)

print(f"共有 {len(all_sectors)} 種不同的 sector：")
print(sorted(all_sectors))

共有 10 種不同的 sector：
['basic materials', 'communication services', 'consumer cyclical', 'consumer defensive', 'energy', 'financial services', 'healthcare', 'industrials', 'real estate', 'technology']


In [233]:
def recommend_as_cold_user(
    model,
    dataset,
    item_features,
    stock_df,
    risk,
    interest_sectors,
    interest_asset_types,
    top_n=10,
    raw_top_k=100
):
    # === 構建冷啟動用戶特徵 ===
    user_traits = [f"risk:{risk.lower()}"]
    user_traits += [f"interest_sector:{s.strip().lower()}" for s in interest_sectors]
    user_traits += [f"interest_asset_type:{t.strip().lower()}" for t in interest_asset_types]

    user_features_temp = dataset.build_user_features(
        [("__cold_user__", user_traits)],
        normalize=False
    )

    num_items = dataset.interactions_shape()[1]
    scores = model.predict(
        user_ids=dataset.mapping()[0]["__cold_user__"],
        item_ids=np.arange(num_items),
        user_features=user_features_temp,
        item_features=item_features
    )

    reverse_item_mapping = {v: k for k, v in dataset.mapping()[2].items()}
    symbol_to_sector = stock_df.set_index("symbol")["sector"].fillna("").str.lower().to_dict()
    symbol_to_asset = stock_df.set_index("symbol")["asset_type"].fillna("").str.lower().to_dict()

    top_indices = np.argsort(-scores)[:raw_top_k]

    etf_list = []
    sector_list = []
    fallback_list = []

    for idx in top_indices:
        symbol = reverse_item_mapping[idx]
        score = scores[idx]
        sector = symbol_to_sector.get(symbol, "")
        asset_type = symbol_to_asset.get(symbol, "")

        if "etf" in interest_asset_types and asset_type == "etf" and len(etf_list) < 5:
            etf_list.append((symbol, score))
        elif sector in interest_sectors:
            sector_list.append((symbol, score))
        else:
            fallback_list.append((symbol, score))

    # 去除重複
    seen = set()
    final = []
    for lst in [etf_list, sector_list, fallback_list]:
        for symbol, score in lst:
            if symbol not in seen:
                final.append((symbol, score))
                seen.add(symbol)
            if len(final) >= top_n:
                break
        if len(final) >= top_n:
            break

    return [sym for sym, _ in final]

In [234]:
recs = recommend_as_cold_user(
    model=best_model,
    dataset=dataset,
    item_features=item_features,
    stock_df=stock_df,
    risk="moderate",
    interest_sectors=["energy"],
    interest_asset_types=["etf","stock"],
    top_n=15
)

print("冷啟動推薦結果：", recs)

冷啟動推薦結果： ['BTC', 'ETH', 'TLT', 'VOO', 'QQQ', 'FET', 'XOM', 'CVX', 'OXY', 'SPY', 'GLD', 'INDA', 'TQQQ', 'LTC', 'SLV']
